# Ingesta de roles de idioma (incremental)

Lee `language_role.json` (JSON multilínea) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Reemplaza la tabla Delta `movie_silver.language_roles` en cada ejecución (`overwrite`), por lo que conserva solo el último lote procesado.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee `language_role.json` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
language_role_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/language_role.json")
    )

In [0]:
language_role_schema = StructType([
    StructField("languageRole", StringType()),
    StructField("roleId", IntegerType())
])

language_role_df = (
    spark.read
        .schema(language_role_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/language_role.json")
)

### Transformación de columnas

Renombra `languageRole` y `roleId` a snake_case y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
language_role_df = (
    language_role_df
        .withColumnsRenamed(
            {
                "languageRole": "language_role",
                "roleId": "role_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env),
                "file_date": lit(v_file_date)
            }
        )
)

language_role_df = add_ingestion_date(language_role_df)

### Escritura en Silver

Reemplaza la tabla Delta `movie_silver.language_roles` (modo `overwrite`).

In [0]:
(
    language_role_df
        .write
            .mode("overwrite")
            .saveAsTable("movie_silver.language_roles")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")